In [2]:
import pandas as pd
import numpy as np
import os

print(os.getcwd())
cnc = pd.read_csv("../../data/raw/cnc/ai4i2020.csv")

print("=== CNC BEFORE ===")
print("Shape:", cnc.shape)
print("Nulls:\n", cnc.isnull().sum())
print("Duplicates:", cnc.duplicated().sum())

/Users/ishekasingh/Documents/u2u-internship-project/reports/notebooks
=== CNC BEFORE ===
Shape: (10000, 14)
Nulls:
 UDI                        0
Product ID                 0
Type                       0
Air temperature [K]        0
Process temperature [K]    0
Rotational speed [rpm]     0
Torque [Nm]                0
Tool wear [min]            0
Machine failure            0
TWF                        0
HDF                        0
PWF                        0
OSF                        0
RNF                        0
dtype: int64
Duplicates: 0


In [3]:
# Drop identifier columns not useful for ML
cnc = cnc.drop(columns=["UDI", "Product ID"])

# Rename columns to clean format
cnc.columns = [
    "type", "air_temp_k", "process_temp_k", "rotational_speed_rpm",
    "torque_nm", "tool_wear_min", "machine_failure",
    "twf", "hdf", "pwf", "osf", "rnf"
]

print("Cleaned columns:", cnc.columns.tolist())

Cleaned columns: ['type', 'air_temp_k', 'process_temp_k', 'rotational_speed_rpm', 'torque_nm', 'tool_wear_min', 'machine_failure', 'twf', 'hdf', 'pwf', 'osf', 'rnf']


In [4]:
print("=== CNC AFTER ===")
print("Shape:", cnc.shape)
print("Nulls:\n", cnc.isnull().sum())
print("Duplicates:", cnc.duplicated().sum())
print(cnc.head())

=== CNC AFTER ===
Shape: (10000, 12)
Nulls:
 type                    0
air_temp_k              0
process_temp_k          0
rotational_speed_rpm    0
torque_nm               0
tool_wear_min           0
machine_failure         0
twf                     0
hdf                     0
pwf                     0
osf                     0
rnf                     0
dtype: int64
Duplicates: 0
  type  air_temp_k  process_temp_k  rotational_speed_rpm  torque_nm  \
0    M       298.1           308.6                  1551       42.8   
1    L       298.2           308.7                  1408       46.3   
2    L       298.1           308.5                  1498       49.4   
3    L       298.2           308.6                  1433       39.5   
4    L       298.2           308.7                  1408       40.0   

   tool_wear_min  machine_failure  twf  hdf  pwf  osf  rnf  
0              0                0    0    0    0    0    0  
1              3                0    0    0    0    0    0  
2     

In [6]:
cnc.to_csv("../../data/processed/cnc_cleaned.csv", index=False)
print("Saved.")

Saved.


In [7]:
wind = pd.read_csv("../../data/raw/wind_turbine/T1.csv")

print("=== WIND TURBINE BEFORE ===")
print("Shape:", wind.shape)
print("Nulls:\n", wind.isnull().sum())
print("Duplicates:", wind.duplicated().sum())
print("Negative power rows:", (wind["LV ActivePower (kW)"] < 0).sum())

=== WIND TURBINE BEFORE ===
Shape: (50530, 5)
Nulls:
 Date/Time                        0
LV ActivePower (kW)              0
Wind Speed (m/s)                 0
Theoretical_Power_Curve (KWh)    0
Wind Direction (°)               0
dtype: int64
Duplicates: 0
Negative power rows: 57


In [8]:
# Convert datetime
wind["Date/Time"] = pd.to_datetime(wind["Date/Time"], format="%d %m %Y %H:%M")

# Remove negative power readings
wind = wind[wind["LV ActivePower (kW)"] >= 0]

# Rename columns
wind.columns = ["datetime", "active_power_kw", "wind_speed_ms", 
                "theoretical_power_kwh", "wind_direction_deg"]

# Engineer failure label
# Power deficit = how far actual power is below theoretical
wind["power_deficit"] = wind["theoretical_power_kwh"] - wind["active_power_kw"]

# Flag as anomaly if deficit is in the top 10% and wind speed is above 3 m/s
# (low wind legitimately produces low power — exclude those)
threshold = wind["power_deficit"].quantile(0.90)
wind["anomaly"] = (
    (wind["power_deficit"] > threshold) & 
    (wind["wind_speed_ms"] > 3)
).astype(int)

print("Anomaly rate:", wind["anomaly"].mean().round(4))

Anomaly rate: 0.1


In [9]:
print("=== WIND TURBINE AFTER ===")
print("Shape:", wind.shape)
print("Nulls:\n", wind.isnull().sum())
print("Anomaly distribution:\n", wind["anomaly"].value_counts())

=== WIND TURBINE AFTER ===
Shape: (50473, 7)
Nulls:
 datetime                 0
active_power_kw          0
wind_speed_ms            0
theoretical_power_kwh    0
wind_direction_deg       0
power_deficit            0
anomaly                  0
dtype: int64
Anomaly distribution:
 anomaly
0    45425
1     5048
Name: count, dtype: int64


In [10]:
wind.to_csv("../../data/processed/wind_cleaned.csv", index=False)
print("Saved.")

Saved.


In [28]:
# CMAPSS has no header and space-separated values
engine = pd.read_csv("../../data/raw/aircraft_engine/CMAPSSData/train_FD001.txt", 
                     sep=r"\s+", header=None)

# Drop last two columns if they are fully null (common CMAPSS issue)
engine = engine.dropna(axis=1, how="all")

print("=== ENGINE BEFORE ===")
print("Shape:", engine.shape)
print("Nulls:\n", engine.isnull().sum())
print("Duplicates:", engine.duplicated().sum())

=== ENGINE BEFORE ===
Shape: (20631, 26)
Nulls:
 0     0
1     0
2     0
3     0
4     0
5     0
6     0
7     0
8     0
9     0
10    0
11    0
12    0
13    0
14    0
15    0
16    0
17    0
18    0
19    0
20    0
21    0
22    0
23    0
24    0
25    0
dtype: int64
Duplicates: 0


In [24]:
engine.columns = [
    "unit_id", "cycle", "op_setting_1", "op_setting_2", "op_setting_3",
    "sensor_1", "sensor_2", "sensor_3", "sensor_4", "sensor_5",
    "sensor_6", "sensor_7", "sensor_8", "sensor_9", "sensor_10",
    "sensor_11", "sensor_12", "sensor_13", "sensor_14", "sensor_15",
    "sensor_16", "sensor_17", "sensor_18", "sensor_19", "sensor_20",
    "sensor_21"
]

In [14]:
# These columns have the same value in every row — useless for ML
zero_var = [col for col in engine.columns if engine[col].std() == 0]
print("Zero variance columns to drop:", zero_var)
engine = engine.drop(columns=zero_var)

Zero variance columns to drop: ['op_setting_3', 'sensor_1', 'sensor_10', 'sensor_18', 'sensor_19']


In [15]:
# RUL = how many cycles remain until the engine fails
# Max cycle per engine unit = the cycle it failed at
max_cycles = engine.groupby("unit_id")["cycle"].max().reset_index()
max_cycles.columns = ["unit_id", "max_cycle"]

engine = engine.merge(max_cycles, on="unit_id")
engine["rul"] = engine["max_cycle"] - engine["cycle"]
engine = engine.drop(columns=["max_cycle"])

print("RUL sample:\n", engine[["unit_id", "cycle", "rul"]].head(10))

RUL sample:
    unit_id  cycle  rul
0        1      1  191
1        1      2  190
2        1      3  189
3        1      4  188
4        1      5  187
5        1      6  186
6        1      7  185
7        1      8  184
8        1      9  183
9        1     10  182


In [16]:
print("=== ENGINE AFTER ===")
print("Shape:", engine.shape)
print("Nulls:\n", engine.isnull().sum())
print("RUL range:", engine["rul"].min(), "to", engine["rul"].max())

=== ENGINE AFTER ===
Shape: (20631, 22)
Nulls:
 unit_id         0
cycle           0
op_setting_1    0
op_setting_2    0
sensor_2        0
sensor_3        0
sensor_4        0
sensor_5        0
sensor_6        0
sensor_7        0
sensor_8        0
sensor_9        0
sensor_11       0
sensor_12       0
sensor_13       0
sensor_14       0
sensor_15       0
sensor_16       0
sensor_17       0
sensor_20       0
sensor_21       0
rul             0
dtype: int64
RUL range: 0 to 361


In [17]:
engine.to_csv("../../data/processed/engine_cleaned.csv", index=False)
print("Saved.")

Saved.


In [18]:
import os

print("Current dir:", os.getcwd())
print("Files here:", os.listdir("."))
print("Parent:", os.listdir(".."))

Current dir: /Users/ishekasingh/Documents/u2u-internship-project/reports/notebooks
Files here: ['00_data_inspection.ipynb', '01_data_cleaning_fixed.ipynb', '.ipynb_checkpoints']
Parent: ['README.md', '.ipynb_checkpoints', 'notebooks']


In [19]:
import os
print(os.getcwd())

/Users/ishekasingh/Documents/u2u-internship-project/reports/notebooks


In [20]:
import pandas as pd

cnc_check = pd.read_csv("../../data/processed/cnc_cleaned.csv")

print(cnc_check.shape)
print(cnc_check.head())
print(cnc_check.isnull().sum())

(10000, 12)
  type  air_temp_k  process_temp_k  rotational_speed_rpm  torque_nm  \
0    M       298.1           308.6                  1551       42.8   
1    L       298.2           308.7                  1408       46.3   
2    L       298.1           308.5                  1498       49.4   
3    L       298.2           308.6                  1433       39.5   
4    L       298.2           308.7                  1408       40.0   

   tool_wear_min  machine_failure  twf  hdf  pwf  osf  rnf  
0              0                0    0    0    0    0    0  
1              3                0    0    0    0    0    0  
2              5                0    0    0    0    0    0  
3              7                0    0    0    0    0    0  
4              9                0    0    0    0    0    0  
type                    0
air_temp_k              0
process_temp_k          0
rotational_speed_rpm    0
torque_nm               0
tool_wear_min           0
machine_failure         0
twf          

In [21]:
wind_check = pd.read_csv("../../data/processed/wind_cleaned.csv")

print(wind_check.shape)
print(wind_check.head())
print(wind_check.isnull().sum())

(50473, 7)
              datetime  active_power_kw  wind_speed_ms  theoretical_power_kwh  \
0  2018-01-01 00:00:00       380.047791       5.311336             416.328908   
1  2018-01-01 00:10:00       453.769196       5.672167             519.917511   
2  2018-01-01 00:20:00       306.376587       5.216037             390.900016   
3  2018-01-01 00:30:00       419.645905       5.659674             516.127569   
4  2018-01-01 00:40:00       380.650696       5.577941             491.702972   

   wind_direction_deg  power_deficit  anomaly  
0          259.994904      36.281117        0  
1          268.641113      66.148316        0  
2          272.564789      84.523429        0  
3          271.258087      96.481664        0  
4          265.674286     111.052276        0  
datetime                 0
active_power_kw          0
wind_speed_ms            0
theoretical_power_kwh    0
wind_direction_deg       0
power_deficit            0
anomaly                  0
dtype: int64


In [26]:
cmapss_check = pd.read_csv("../../data/processed/engine_cleaned.csv")

print(cmapss_check.shape)
print(cmapss_check.head())
print(cmapss_check.isnull().sum())

(20631, 22)
   unit_id  cycle  op_setting_1  op_setting_2  sensor_2  sensor_3  sensor_4  \
0        1      1       -0.0007       -0.0004    641.82   1589.70   1400.60   
1        1      2        0.0019       -0.0003    642.15   1591.82   1403.14   
2        1      3       -0.0043        0.0003    642.35   1587.99   1404.20   
3        1      4        0.0007        0.0000    642.35   1582.79   1401.87   
4        1      5       -0.0019       -0.0002    642.37   1582.85   1406.22   

   sensor_5  sensor_6  sensor_7  ...  sensor_11  sensor_12  sensor_13  \
0     14.62     21.61    554.36  ...      47.47     521.66    2388.02   
1     14.62     21.61    553.75  ...      47.49     522.28    2388.07   
2     14.62     21.61    554.26  ...      47.27     522.42    2388.03   
3     14.62     21.61    554.45  ...      47.13     522.86    2388.08   
4     14.62     21.61    554.00  ...      47.28     522.19    2388.04   

   sensor_14  sensor_15  sensor_16  sensor_17  sensor_20  sensor_21  rul  